<a href="https://colab.research.google.com/github/grasht/projects_ML_HW_4/blob/main/HW4_pt2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 2
Dataset: Rossmann Store Sales.

Problem: Forecast sales using store, promotion, and competitor data.

Provided By: Pratyusha Kar at Kagglehub.

Link: https://www.kaggle.com/datasets/pratyushakar/rossmann-store-sales?select=train.csv

This dataset and problem require sequence models because sales are sequential and treating the sales data as it consists of independent rows would cause us to miss important trends. For example, a sale after a busy holiday period might perform worse than a sale after a quiet week. Any trends in the data such as this could not be captured without sequence models.

In [197]:
#Part 1

import kagglehub
from kagglehub import KaggleDatasetAdapter

file_path = "test.csv"

# Load the latest version
test = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "pratyushakar/rossmann-store-sales",
  file_path,
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

file_path = "train.csv"
train = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "pratyushakar/rossmann-store-sales",
  file_path,
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("First 5 records:", train.head())
print("First 5 records:", test.head())

/tmp/ipykernel_1944/307982777.py:9: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  test = kagglehub.load_dataset(


Using Colab cache for faster access to the 'rossmann-store-sales' dataset.


/tmp/ipykernel_1944/307982777.py:20: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  train = kagglehub.load_dataset(


Using Colab cache for faster access to the 'rossmann-store-sales' dataset.
First 5 records:    Store  DayOfWeek        Date  Sales  Customers  Open  Promo StateHoliday  \
0      1          5  2015-07-31   5263        555     1      1            0   
1      2          5  2015-07-31   6064        625     1      1            0   
2      3          5  2015-07-31   8314        821     1      1            0   
3      4          5  2015-07-31  13995       1498     1      1            0   
4      5          5  2015-07-31   4822        559     1      1            0   

   SchoolHoliday  
0              1  
1              1  
2              1  
3              1  
4              1  
First 5 records:    Id  Store  DayOfWeek        Date  Open  Promo StateHoliday  SchoolHoliday
0   1      1          4  2015-09-17   1.0      1            0              0
1   2      3          4  2015-09-17   1.0      1            0              0
2   3      7          4  2015-09-17   1.0      1            0          

/usr/local/lib/python3.12/dist-packages/kagglehub/pandas_datasets.py:91: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  result = read_function(


In [198]:
#Part 1 - Preprocessing
import torch
from torch import nn
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

print(torch.__version__)
print("CUDA available:", torch.cuda.is_available()) # Checks for GPU access
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device.")

#Map Correlations

feature_names = train.columns.tolist()

print(feature_names)

features = train.drop('Sales', axis=1).drop('Date', axis=1)
target = train['Sales'] # Assign 'Sales' column to target

#Learn on Store 1
features = features[features["Store"] == 1]
target = train[train["Store"] == 1]["Sales"]

X = features.copy()

#Encoding with pandas dummies
X = pd.get_dummies(X, drop_first=True)
X = X.apply(pd.to_numeric, errors='coerce').astype(float)
X = X.replace([np.inf, -np.inf], np.nan)

# Impute NaN values with the mean of each column.
# For columns that are entirely NaN, their mean will be NaN. Fill those with 0.
X = X.fillna(X.mean(numeric_only=True)).fillna(0)



2.10.0+cu128
CUDA available: True
Using cuda device.
['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday']


In [199]:
#Part 1 - Sequencing and More Preprocessing

#Squence Length
#n_days = 3
n_days = 7
#n_days = 30
#n_days = 60

X_numeric = X.values.astype(float)
target_numeric = target.values.astype(float)

def create_sequences(data, target, seq_len):
    X_seq = []
    y_seq = []
    for i in range(len(data) - seq_len):
        X_seq.append(data[i:i+seq_len])
        y_seq.append(target[i+seq_len])
    return np.array(X_seq), np.array(y_seq)

X_seq, y_seq = create_sequences(X_numeric, target_numeric, n_days)

# Train/test split
split = int(len(X_seq) * 0.8)
X_train, X_test = X_seq[:split], X_seq[split:]
y_train, y_test = y_seq[:split], y_seq[split:]

# Scaling
samples, seq_len, n_features = X_train.shape
scaler = StandardScaler()

X_train_2D = X_train.reshape(-1, n_features)
X_test_2D = X_test.reshape(-1, n_features)

X_train_2D = scaler.fit_transform(X_train_2D)
X_test_2D = scaler.transform(X_test_2D)

# Reshape back to 3D
X_train = X_train_2D.reshape(samples, seq_len, n_features)
X_test = X_test_2D.reshape(X_test.shape[0], seq_len, n_features)

In [200]:
#Part 1 - Class Definition And Data Loaders
class RNNModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
      out, _ = self.rnn(x)
      out = self.fc(out[:, -1, :])
      return out

input_size = X_seq.shape[2]
hidden_size = 20
#num_layers = 2
num_layers = 1
#num_layers = 3

model = RNNModel(input_size, hidden_size, num_layers).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=64, shuffle=False)


In [201]:
#Part 1/2 - Train and Test loops
def train_loop(dataloader, model, loss_fn, optimizer, verbose=False):
  model.train()
  for X, y in dataloader:
    X = X.to(device)
    y = y.to(device).float().unsqueeze(1)

    pred = model(X)
    loss = loss_fn(pred, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if verbose:
      print(f"loss: {loss.item():>7f}")

def test_loop(dataloader, model, loss_fn, verbose):
  model.eval()
  total_loss = 0

  with torch.no_grad():
    for X, y in dataloader:
      X = X.to(device)
      y = y.to(device).float().unsqueeze(1)

      pred = model(X)
      loss = loss_fn(pred, y)
      total_loss += loss.item() * X.size(0)

  avg_loss = total_loss / len(dataloader.dataset)  # mean loss per sample
  if verbose:
    print(f"Test Loss: {avg_loss:.4f}")
  return avg_loss

In [202]:
#Part 1 - Eval
epochs = 100
for t in range(epochs):
    verbose = (t+1) % int(epochs / 10) == 0
    if verbose:
      print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, torch.optim.Adam(model.parameters(), lr=0.001), verbose)
    test_loop(test_dataloader, model, loss_fn, verbose)

Epoch 10
-------------------------------
loss: 23034658.000000
loss: 15556298.000000
loss: 17107010.000000
loss: 19749516.000000
loss: 16964432.000000
loss: 17689574.000000
loss: 20901152.000000
loss: 20675696.000000
loss: 18253480.000000
loss: 21026868.000000
loss: 18398396.000000
loss: 18722820.000000
Test Loss: 21819677.3476
Epoch 20
-------------------------------
loss: 14941331.000000
loss: 19151986.000000
loss: 22861908.000000
loss: 16831436.000000
loss: 17405584.000000
loss: 18488998.000000
loss: 21395328.000000
loss: 20199850.000000
loss: 18536140.000000
loss: 16479726.000000
loss: 20772184.000000
loss: 21696218.000000
Test Loss: 21797225.5936
Epoch 30
-------------------------------
loss: 17196080.000000
loss: 16550274.000000
loss: 18122292.000000
loss: 20231272.000000
loss: 20162736.000000
loss: 20908642.000000
loss: 19545750.000000
loss: 18248162.000000
loss: 19389428.000000
loss: 19318906.000000
loss: 20205092.000000
loss: 17275620.000000
Test Loss: 21776178.6845
Epoch 40
-

In [203]:
#Part 2 - Class Definitions
class LSTM_RNN(nn.Module):
  def __init__(self, input_size, hidden_size, num_layers):
    super().__init__()
    self.rnn = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
    self.fc = nn.Linear(hidden_size, 1)

  def forward(self, x):
    out, (h_n, c_n) = self.rnn(x)
    out = self.fc(out[:, -1, :])
    return out

class GRU_RNN(nn.Module):
  def __init__(self, input_size, hidden_size, num_layers):
    super().__init__()
    self.rnn = nn.GRU(input_size, hidden_size, num_layers, batch_first=True)
    self.fc = nn.Linear(hidden_size, 1)

  def forward(self, x):
    out, h_n = self.rnn(x)
    out = self.fc(out[:, -1, :])
    return out

In [204]:
#Part 2 - Eval LSTM
model = LSTM_RNN(input_size, hidden_size, num_layers).to(device)
epochs = 100
for t in range(epochs):
    verbose = (t+1) % 10 == 0
    if verbose:
      print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, torch.optim.Adam(model.parameters(), lr=0.001), verbose)
    test_loop(test_dataloader, model, loss_fn, verbose)

Epoch 10
-------------------------------
loss: 16442850.000000
loss: 19312260.000000
loss: 19463768.000000
loss: 18120404.000000
loss: 18983034.000000
loss: 17476824.000000
loss: 20828286.000000
loss: 17585804.000000
loss: 18375760.000000
loss: 22484272.000000
loss: 20049756.000000
loss: 19188300.000000
Test Loss: 21827209.0053
Epoch 20
-------------------------------
loss: 17506060.000000
loss: 18744852.000000
loss: 20892656.000000
loss: 18853096.000000
loss: 16322479.000000
loss: 19422912.000000
loss: 20379216.000000
loss: 20814612.000000
loss: 18742840.000000
loss: 20188372.000000
loss: 17214616.000000
loss: 18819386.000000
Test Loss: 21800733.0267
Epoch 30
-------------------------------
loss: 16702229.000000
loss: 18588320.000000
loss: 19432604.000000
loss: 18217640.000000
loss: 20527852.000000
loss: 18632844.000000
loss: 16714020.000000
loss: 22683194.000000
loss: 21627512.000000
loss: 20658236.000000
loss: 17714200.000000
loss: 14960606.000000
Test Loss: 21779119.7540
Epoch 40
-

In [205]:
#Part 2 - Eval GRU
model = GRU_RNN(input_size, hidden_size, num_layers).to(device)
epochs = 100
for t in range(epochs):
    verbose = (t+1) % 10 == 0
    if verbose:
      print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, torch.optim.Adam(model.parameters(), lr=0.001), verbose)
    test_loop(test_dataloader, model, loss_fn, verbose)

Epoch 10
-------------------------------
loss: 18184368.000000
loss: 17947094.000000
loss: 19356504.000000
loss: 19042930.000000
loss: 18462588.000000
loss: 21266364.000000
loss: 19692884.000000
loss: 17997976.000000
loss: 18075594.000000
loss: 18484184.000000
loss: 20400448.000000
loss: 19484356.000000
Test Loss: 21825156.6417
Epoch 20
-------------------------------
loss: 19074610.000000
loss: 15931214.000000
loss: 20507460.000000
loss: 19773712.000000
loss: 20756408.000000
loss: 20371144.000000
loss: 17451240.000000
loss: 16857436.000000
loss: 17806382.000000
loss: 19134160.000000
loss: 21218228.000000
loss: 19063440.000000
Test Loss: 21797889.5187
Epoch 30
-------------------------------
loss: 18454312.000000
loss: 19790492.000000
loss: 18022068.000000
loss: 18296376.000000
loss: 17838034.000000
loss: 22549986.000000
loss: 20694354.000000
loss: 18677514.000000
loss: 16009632.000000
loss: 19084468.000000
loss: 18561678.000000
loss: 20043056.000000
Test Loss: 21776780.2567
Epoch 40
-

# Part 2 - Results
(100 Epochs)

**3 Day Sequences**
Type | Final Test Loss
-----|----------------
Vanilla RNN | 21513824.1702
LSTM RNN | 21517239.5319
GRU RNN | 21517561.9574

**7 Day Sequences**
Type | Final Test Loss
-----|----------------
Vanilla RNN | 21625245.2941
LSTM RNN | 21631606.9519
GRU RNN | 21631289.2193

**30 Day Sequences**
Type | Final Test Loss
-----|----------------
Vanilla RNN | 21516627.0492
LSTM RNN | 21513186.2514
GRU RNN | 21513887.0710

**60 Day Sequences**
Type | Final Test Loss
-----|----------------
Vanilla RNN | 21509375.6610
LSTM RNN | 21505065.9548
GRU RNN | 21501510.0000


The results show marginal improvements when using LSTM and GRU. The improvements may have been more drastic if the dataset wasn't reduced to just one store (limiting the overall amount of data). However, the improvements are the most drastic in the test with the largest sequences showcasing the ability of the LSTM and the GRU to better learn trends across larger time periods.

Surprisingly, the 3 day sequence outperformed the 7 and 30 day sequences, suggesting that, for the store 1 data, short term trends were not as impactful as longer term trends (60 days or more). Another possible explination is that 2 hidden layers is overkill and leading to overfitting for lower day sequences causing noise to impact results.To test this theory, I tested 3 days with 1 hidden layer and 60 days with 3 and these were the results.

**3 Days, 1 Layer:**
Days|Vanilla | LSTM | GRU
--|------|------|-----
3|21518011.5745 | 21516827.0213 | 21515119.8298
7|21633525.0695 | 21632475.4652 | 21630312.0214

These results suggest the former explination is more likely to be the case.

I also wanted to see if 60 day sequences required more layers to capture more complex results.

**60 Days, 3 Layers:**
Vanilla | LSTM | GRU
--------|------|-----
21504710.9718 | 21502524.0113 | 21506520.9492

The only model to perform worse here was GRU. This makes sense as it has simpler gates and fewer parameters than LSTM. The extra depth likely caused the model to distort or forget actually useful information.



# Part 3
The traditional feed forward network can be used to solve the same problem, but with limited learning abilities. Certain features are included in this data specifically to allow for the learning of certain trends that only exist when the data is placed into proper sequences. When the rows are treated as independant data points, a model can not pick up on aspects such as how a promotional period in prior days might impact the next days sales. The data is provided in such a way that it is possible to treat the rows independant of one another, but it is not ideal.

These types of decisions have implications on the learning. Even limiting the problem scope to just Store 1 causes the model to be less generic and to miss out on interstore trends, but that simplified sequencing and was adequate for the purposes of experimenting with RNNs.